# Automathon sur Databricks — prise en main

Ce notebook installe l'Automathon 1.0.0 sur Databricks (le simulateur sans affichage, le Headless, et le kit
Python `PythonAI`), vérifie qu'il fonctionne, mesure la vitesse de la machine, puis entraîne un agent par
renforcement (PPO), l'évalue et vous permet de le récupérer sur votre PC.

**Avant de commencer :**
1. Un Volume Unity Catalog existe, par exemple `/Volumes/workspace/default/automathon`.
2. Le kit Linux a été déposé dans ce Volume :
   - `Automathon_1.0.0_Linux-ARM-databricks.zip` pour une machine à processeur ARM (AWS Graviton) ;
   - `Automathon_1.0.0_Linux-x86-64.zip` pour une machine x86-64.

   Dans le doute, déposez les deux : le notebook choisit celui qui correspond à la machine.
3. Le notebook est attaché à une ressource de calcul (bouton *Connect* en haut à droite).

Exécutez les cellules **dans l'ordre**, une par une (`Maj + Entrée`). Après un redémarrage de la session (ou le
lendemain), repartez de l'étape 1.

**Rappel des règles du jeu** (détail dans `PythonAI/README.md`) : duel de vaisseaux, décisions à 15 Hz (une frame sur
4), balles à 60 dégâts, missile armé après 1 unité de vol, zone sûre qui se referme de 10 s à 40 s (50 PV/s dehors).

## Étape 1 — Installer les bibliothèques Python

`torch` est installé en version **CPU** : le PPO du kit utilise un petit réseau, et le calcul est dominé par la
simulation, qui tourne sur CPU. Les versions sont celles du kit (`uv.lock`) : un modèle entraîné ici se charge sans
souci sur les PC Windows, Linux et macOS. L'installation prend 1 à 3 minutes.

In [ ]:
%pip install --quiet torch==2.14.1 --index-url https://download.pytorch.org/whl/cpu

In [ ]:
%pip install --quiet stable-baselines3==2.9.2 gymnasium==1.4.0 numba==0.68.1 pydantic pyzmq tqdm "hydra-core>=1.3.2" tensorboard

Redémarrer Python pour charger les bibliothèques installées (les variables du notebook sont effacées, c'est
normal).

In [ ]:
dbutils.library.restartPython()

## Étape 2 — Paramètres, décompression et variables d'environnement

**À adapter :** `VOLUME` si votre Volume porte un autre nom, `EQUIPE` avec le nom de votre équipe.

Le kit est décompressé sur le disque local de la machine (`/tmp`), car le Headless doit s'exécuter depuis un disque
local. Ce disque est effacé quand la session s'arrête : tout ce qui doit être gardé (checkpoints, modèles, vos
agents) est enregistré dans le Volume.

In [ ]:
import os, sys, glob, shutil, zipfile, subprocess, datetime, threading, time, platform, re

# ---- À adapter ---------------------------------------------------------------
VOLUME = "/Volumes/workspace/default/automathon"   # Volume où se trouve le kit
EQUIPE = "mon_equipe"                              # nom de votre équipe / de votre agent
# -------------------------------------------------------------------------------

VERSION = "1.0.0"
ARCH = platform.machine()
ZIP_NAME = {"aarch64": f"Automathon_{VERSION}_Linux-ARM-databricks.zip", "arm64": f"Automathon_{VERSION}_Linux-ARM-databricks.zip",
            "x86_64": f"Automathon_{VERSION}_Linux-x86-64.zip", "AMD64": f"Automathon_{VERSION}_Linux-x86-64.zip"}.get(ARCH)
if ZIP_NAME is None:
    raise RuntimeError(f"Processeur {ARCH} non pris en charge (ARM64 ou x86-64 attendus).")
if not os.path.exists(f"{VOLUME}/{ZIP_NAME}"):
    raise FileNotFoundError(f"Processeur {ARCH} : déposez {ZIP_NAME} dans {VOLUME}.")

LOCAL_DIR = "/tmp/automathon"
ROOT      = f"{LOCAL_DIR}/{ZIP_NAME.removesuffix('.zip')}"   # contient Headless/ et PythonAI/
PYTHONAI  = f"{ROOT}/PythonAI"
HEADLESS  = f"{ROOT}/Headless/Headless"

if not os.path.isdir(PYTHONAI):
    print("Décompression de", ZIP_NAME, "...")
    zipfile.ZipFile(f"{VOLUME}/{ZIP_NAME}").extractall(LOCAL_DIR)
os.chmod(HEADLESS, 0o755)                            # le droit d'exécution peut être perdu à la décompression
os.environ["AUTOMATHON_HEADLESS_PATH"] = HEADLESS    # trouvé aussi sans, car Headless/ est à côté de PythonAI/

# Dossiers persistants dans le Volume
for d in ("checkpoints", "modeles", "exports", "mes_agents"):
    os.makedirs(f"{VOLUME}/{d}", exist_ok=True)

# Vos agents sont gardés dans le Volume et recopiés dans PythonAI/agents/ à chaque session
for agent_dir in glob.glob(f"{VOLUME}/mes_agents/*/"):
    shutil.copytree(agent_dir, f"{PYTHONAI}/agents/{os.path.basename(agent_dir.rstrip('/'))}", dirs_exist_ok=True)

# Un Volume ne sait pas écrire un zip directement (Errno 5) : l'entraînement écrit ses checkpoints sur le disque
# local, et un fil en arrière-plan les recopie dans le Volume toutes les minutes.
CKPT_DIR    = f"{LOCAL_DIR}/checkpoints/{EQUIPE}"   # écrit par l'entraînement
CKPT_VOLUME = f"{VOLUME}/checkpoints/{EQUIPE}"      # copie de sauvegarde, conservée
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(CKPT_VOLUME, exist_ok=True)
os.environ["AUTOMATHON_CHECKPOINT_DIR"] = CKPT_DIR


def sync_checkpoints():
    # Recopie dans le Volume les checkpoints nouveaux ou modifiés (terminés depuis plus de 10 s).
    for src in glob.glob(f"{CKPT_DIR}/*.zip"):
        dst = f"{CKPT_VOLUME}/{os.path.basename(src)}"
        if time.time() - os.path.getmtime(src) < 10:
            continue
        if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src):
            shutil.copyfile(src, dst)


def _sync_loop():
    while True:
        try:
            sync_checkpoints()
        except Exception as e:
            print("[sauvegarde] erreur :", e)
        time.sleep(60)


if not any(t.name == "sync_checkpoints" for t in threading.enumerate()):
    threading.Thread(target=_sync_loop, name="sync_checkpoints", daemon=True).start()

# Nombre de Headless en parallèle : 2 par CPU, entre 8 et 32 (un Headless attend Python une partie du temps,
# voir la section 14 du README). La taille des rollouts reste de 8 192 steps : n_steps x NUM_ENVS.
CPUS = len(os.sched_getaffinity(0))
NUM_ENVS = max(8, min(2 * CPUS, 32))
N_STEPS = 8192 // NUM_ENVS
TRAIN_ARGS = [f"unity.num_envs={NUM_ENVS}", f"ppo.n_steps={N_STEPS}"]
os.environ["AUTOMATHON_STARTUP_WAIT"] = "30"         # machine cloud parfois lente au démarrage


def run(*args):
    # Lance un script de PythonAI/ (comme dans un terminal) et affiche sa sortie en direct.
    # PYTHONPATH = chemins du notebook : sans cela, le script ne voit pas les bibliothèques installées par %pip.
    env = {**os.environ, "PYTHONPATH": os.pathsep.join(p for p in sys.path if p)}
    p = subprocess.Popen([sys.executable, "-u", *args], cwd=PYTHONAI, text=True, bufsize=1, env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    time.sleep(11)
    sync_checkpoints()                               # dernière sauvegarde, même après une erreur
    if p.returncode != 0:
        raise RuntimeError(f"Le script s'est arrêté avec le code {p.returncode}")


print("Kit     :", ZIP_NAME)
print("Python  :", sys.version.split()[0], "| CPU utilisables :", CPUS, "| processeur :", ARCH)
print("Entraînement :", NUM_ENVS, "Headless en parallèle, n_steps =", N_STEPS)
print("Agents  :", sorted(a for a in os.listdir(f"{PYTHONAI}/agents") if not a.endswith(".py")))

## Étape 3 — Caractéristiques de la machine

Processeur, nombre de cœurs, quota CPU et mémoire. Le débit d'entraînement dépend surtout de la vitesse d'un cœur
et du nombre de cœurs (voir la section 14 de `PythonAI/README.md`).

In [ ]:
info = open("/proc/cpuinfo").read()
parts = set(re.findall(r"CPU part\s*:\s*(0x[0-9a-f]+)", info))
graviton = {"0xd0c": "Neoverse N1 → AWS Graviton2", "0xd40": "Neoverse V1 → AWS Graviton3",
            "0xd4f": "Neoverse V2 → AWS Graviton4"}
for p in parts:
    print("Cœur ARM  :", p, "=", graviton.get(p, "modèle inconnu"))
model = re.search(r"model name\s*:\s*(.+)", info)
if model:
    print("Processeur:", model.group(1))
print("CPU vus   :", os.cpu_count(), "| utilisables :", CPUS)
for f in ("/sys/fs/cgroup/cpu.max", "/sys/fs/cgroup/cpu/cpu.cfs_quota_us"):
    if os.path.exists(f):
        print("Quota CPU :", open(f).read().strip(), f"({f})")
mem = re.search(r"MemTotal:\s*(\d+)", open("/proc/meminfo").read())
print("Mémoire   :", round(int(mem.group(1)) / 1024**2, 1), "Go")

## Étape 4 — Diagnostic : le Headless démarre-t-il ?

Lance un Headless quelques secondes et vérifie qu'il écoute sur son port, puis lance le test rapide du kit
(quelques matchs scriptés, se termine par `OK`).

In [ ]:
import socket

port = 5999
env = {**os.environ, "DOTNET_SYSTEM_GLOBALIZATION_INVARIANT": "1", "DOTNET_gcServer": "0"}
proc = subprocess.Popen([HEADLESS, f"tcp://127.0.0.1:{port}"], cwd=ROOT, env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
ok = False
for _ in range(30):
    time.sleep(1)
    if proc.poll() is not None:
        break
    with socket.socket() as s:
        if s.connect_ex(("127.0.0.1", port)) == 0:
            ok = True
            break
proc.kill()
out = proc.communicate()[0]
if ok:
    print("OK : le Headless écoute sur le port", port)
    run("../tools/smoke_test.py")
else:
    print("ÉCHEC. Code de sortie :", proc.returncode, "\n--- sortie du Headless ---\n", out[-3000:])

## Étape 5 — Premier duel entre deux bots

10 matchs de `rush_bot` contre `dummy_bot` sur le Headless, dans les conditions d'un match (quelques secondes).

In [ ]:
run("evaluation/run_matchup.py", "rush_bot", "dummy_bot", "--games", "10")

## Étape 6 — Mesure du débit de la machine

Un court entraînement de 300 000 steps, pour connaître la vitesse de cette machine en steps par
seconde (un step = une décision de l'agent = 4 frames de jeu). Le débit s'affiche aussi pendant tout entraînement
(ligne `fps`).

In [ ]:
start = time.time()
run("training/rl_training.py", "total_timesteps=300000", *TRAIN_ARGS,
    f"save.model={LOCAL_DIR}/mesure_debit.zip", f"save.checkpoint_dir={LOCAL_DIR}/mesure_debit_ckpt")
elapsed = time.time() - start
print(f"\n~{300000 / elapsed:,.0f} steps/s (démarrage compris) : 3 M de steps en ~{3e6 / (300000 / elapsed) / 60:.0f} min".replace(",", " "))

## Étape 7 — (Optionnel) Écrire son propre agent

Modifiez le code ci-dessous puis exécutez la cellule : l'agent est enregistré dans le Volume
(`mes_agents/<EQUIPE>/agent.py`, donc conservé) et copié dans `PythonAI/agents/`. Positions et vitesses sont en
unités Unity (l'arène fait 27,4 × 15,4), la zone sûre est dans `state.ZoneHalfSize` : voir le README du kit.

In [ ]:
AGENT_CODE = """
from bridge.datatypes import GameState, AIAction, FixedVector2


class ParticipantAgent:
    def get_action(self, state: GameState) -> AIAction:
        if not state.SelfShip or not state.EnemyShip:
            return AIAction()
        to_enemy = state.EnemyShip.Position - state.SelfShip.Position
        aim = to_enemy.normalized() if to_enemy.length_squared() > 0 else FixedVector2(1, 0)
        return AIAction(
            MovingDirection=aim,
            AimingDirection=aim,
            MachineGun=state.SelfShip.MachineGunCooldownFramesLeft == 0,
        )
"""

for dest in (f"{VOLUME}/mes_agents/{EQUIPE}", f"{PYTHONAI}/agents/{EQUIPE}"):
    os.makedirs(dest, exist_ok=True)
    with open(f"{dest}/agent.py", "w") as f:
        f.write(AGENT_CODE.lstrip())
print("Agent enregistré :", EQUIPE)

run("evaluation/run_matchup.py", EQUIPE, "rush_bot", "--games", "10")

## Étape 8 — Entraîner un agent par renforcement

L'entraînement d'exemple du kit : un PPO from scratch contre `rush_bot`, avec les réglages par défaut
(`training/conf/rl_training.yaml`), 3 millions de steps. La durée dépend de la machine (voir l'étape 6). Les checkpoints sont
recopiés dans le Volume (`checkpoints/<EQUIPE>/`) toutes les minutes et à la fin du run.

Tous les réglages se changent en ajoutant des arguments Hydra, par exemple `"total_timesteps=10000000"`,
`"ppo.learning_rate=1e-4"` ou `"opponent=dummy_bot"` (section 10 du README). Pour un essai rapide, mettez
`"total_timesteps=200000"`.

In [ ]:
run("training/rl_training.py", *TRAIN_ARGS)

## Étape 9 — Évaluer le modèle

Taux de victoire contre `rush_bot` sur 50 cartes fixes jouées des deux côtés (100 matchs), avec son intervalle de
confiance, puis comment il gagne : dégâts par arme et par distance, durée des matchs, dégâts de zone.

In [ ]:
run("evaluation/evaluate_model.py", "training/rl_model.zip", "--opponent", "rush_bot", "--maps", "50", "--backend", "binary")
run("evaluation/match_stats.py", "model:training/rl_model.zip", "rush_bot", "--games", "20")

## Étape 10 — Sauvegarder le modèle et préparer l'agent

Copie le modèle dans le Volume (`modeles/`), en fait l'agent rendu (`mes_agents/my_rl_agent/` : le code de
`my_rl_agent` et le modèle en `model.zip`), et crée l'archive portable (`export_agent.py`) dans `exports/`.
Pour récupérer un fichier sur votre PC : **Catalog** → votre Volume → dossier → menu `⋮` du fichier →
**Download**.

In [ ]:
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
shutil.copyfile(f"{PYTHONAI}/training/rl_model.zip", f"{VOLUME}/modeles/rl_model_{stamp}.zip")
print("Modèle :", f"{VOLUME}/modeles/rl_model_{stamp}.zip")

agent_dir = f"{VOLUME}/mes_agents/my_rl_agent"
os.makedirs(agent_dir, exist_ok=True)
for name in ("agent.py", "feature_extractor.py"):
    shutil.copyfile(f"{PYTHONAI}/agents/my_rl_agent/{name}", f"{agent_dir}/{name}")
shutil.copyfile(f"{PYTHONAI}/training/rl_model.zip", f"{agent_dir}/model.zip")
print("Agent  :", agent_dir)

run("training/export_agent.py", "my_rl_agent")
latest = max(glob.glob(f"{PYTHONAI}/training/mon_agent_*.zip"), key=os.path.getmtime)
shutil.copyfile(latest, f"{VOLUME}/exports/{os.path.basename(latest)}")
print("Export :", f"{VOLUME}/exports/{os.path.basename(latest)}")

## Étape 11 — Entraînements longs et reprise

- Une session peut s'arrêter (inactivité, quota) : les checkpoints sont dans `checkpoints/<EQUIPE>/` du Volume.
  Pour **reprendre** depuis le dernier, utilisez la cellule ci-dessous (`init.model` charge le checkpoint au lieu
  de partir de zéro ; ajustez `total_timesteps`).
- Pour un run de plusieurs heures sans garder l'onglet ouvert : **Schedule** (en haut à droite du notebook) →
  *Run now* / tâche planifiée, qui exécute le notebook entier sous forme de Job.
- Un agent RL peut stagner longtemps avant de progresser d'un coup : suivez aussi les dégâts infligés et la durée
  des matchs (étape 9), pas seulement le taux de victoire.

In [ ]:
# Cherche dans le Volume (sessions précédentes) et sur le disque local (session en cours)
ckpts = sorted(glob.glob(f"{CKPT_VOLUME}/*.zip") + glob.glob(f"{CKPT_DIR}/*.zip"), key=os.path.getmtime)
print("Dernier checkpoint :", ckpts[-1] if ckpts else "aucun")
if ckpts:
    run("training/rl_training.py", "total_timesteps=3000000", *TRAIN_ARGS, f"init.model={ckpts[-1]}")

## Utiliser l'agent sur son PC

1. Téléchargez le dossier `mes_agents/my_rl_agent/` du Volume (ou l'archive de `exports/`).
2. Copiez-le dans `PythonAI/agents/my_rl_agent/` du kit de votre PC (le modèle est `model.zip`, à côté de
   `agent.py`).
3. `uv run python agents/run_my_agent.py my_rl_agent --launch-game`, puis dans le jeu : vaisseau en **AI**, port
   `5555`, **PLAY**.